# Skin color filter

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

import PIL
from PIL import Image

import random
import os


In [ ]:
def detect_skin_regions(image_path, save_path="skin.png", show_images=True):
    # Load image with PIL and convert to OpenCV BGR format
    image = Image.open(image_path)
    image = cv2.cvtColor(np.array(image), cv2.COLOR_RGB2BGR)

    # Define YCrCb skin color range
    min_YCrCb = np.array([0, 133, 77], np.uint8)
    max_YCrCb = np.array([235, 173, 127], np.uint8)

    # Convert to YCrCb color space and apply skin mask
    imageYCrCb = cv2.cvtColor(image, cv2.COLOR_BGR2YCrCb)
    skinRegionYCrCb = cv2.inRange(imageYCrCb, min_YCrCb, max_YCrCb)
    skinYCrCb = cv2.bitwise_and(image, image, mask=skinRegionYCrCb)

    # Save side-by-side image
    if save_path:
        combined = np.hstack([image, skinYCrCb])
        cv2.imwrite(save_path, combined)

    # Optionally show images
    if show_images:
        plt.figure(figsize=(10, 5))
        plt.subplot(1, 2, 1)
        plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
        plt.title('Original Image')
        plt.axis('off')

        plt.subplot(1, 2, 2)
        plt.imshow(cv2.cvtColor(skinYCrCb, cv2.COLOR_BGR2RGB))
        plt.title('Skin Detected Image')
        plt.axis('off')
        plt.show()

    return image, skinYCrCb



In [ ]:
image_path = r"C:\Users\propietari\sintesisII\14.08.24-15.08.24\no_detection_HM20240814231919_VIS.jpeg"
original_img, skin_img = detect_skin_regions(image_path, save_path="skin_detected.png", show_images=True)

In [ ]:
image_path = r"C:\Users\propietari\sintesisII\no_detection\no_detection_HM20241111044658.VIS.jpeg"
original_img, skin_img = detect_skin_regions(image_path, save_path="skin_detected.png", show_images=True)

In [ ]:
image_path = r"C:\Users\propietari\sintesisII\no_detection\no_detection_HM20240912140105.VIS.jpeg"
original_img, skin_img = detect_skin_regions(image_path, save_path="skin_detected.png", show_images=True)

In [ ]:
image_path = r"C:\Users\propietari\sintesisII\no_detection\no_detection_HM20240612031748.VIS.jpeg"
original_img, skin_img = detect_skin_regions(image_path, save_path="skin_detected.png", show_images=True)

In [ ]:
# Load image with PIL and convert to OpenCV format
image_path = r"C:\Users\propietari\sintesisII\14.08.24-15.08.24\no_detection_HM20240814231919_VIS.jpeg"
image = Image.open(image_path)
image = cv2.cvtColor(np.array(image), cv2.COLOR_RGB2BGR)

# skin detection in YCrCb
min_YCrCb = np.array([0, 133, 77], np.uint8)
max_YCrCb = np.array([235, 173, 127], np.uint8)

imageYCrCb = cv2.cvtColor(image, cv2.COLOR_BGR2YCR_CB)
skinMask = cv2.inRange(imageYCrCb, min_YCrCb, max_YCrCb)

# Remove small islands using morphological operations
kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
skinMask = cv2.morphologyEx(skinMask, cv2.MORPH_OPEN, kernel, iterations=25) # Removes small blobs
skinMask = cv2.morphologyEx(skinMask, cv2.MORPH_CLOSE, kernel, iterations=25) # Closes small holes

num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(skinMask, connectivity=8)
min_area = 2000 
for i in range(1, num_labels):
    if stats[i, cv2.CC_STAT_AREA] < min_area:
        skinMask[labels == i] = 0

# apply refined mask
skinYCrCb = cv2.bitwise_and(image, image, mask=skinMask)

# Save comparison output
cv2.imwrite("skin_cleaned.png", np.hstack([image, skinYCrCb]))

#show the results
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
plt.title('Original Image')
plt.axis('off')
plt.subplot(1, 2, 2)
plt.imshow(cv2.cvtColor(skinYCrCb, cv2.COLOR_BGR2RGB))
plt.title('Refined Skin Detected Image')
plt.axis('off')
plt.show()
